# 04 — Modèles de deep learning (LSTM et DistilBERT)

**Objectif de ce notebook** : entraîner deux modèles de **deep learning** et les comparer aux modèles classiques du notebook 03 :
- un **LSTM bidirectionnel**, réseau de neurones récurrent qui lit le texte mot par mot ;
- **DistilBERT**, un modèle de langage pré-entraîné (*transformer*) qu'on spécialise sur nos catégories (*fine-tuning*).

Mêmes critères que le notebook 03 : accuracy, F1 macro, temps d'entraînement et taille du modèle.

⚠️ Ce notebook est fait pour **Google Colab avec un GPU** : *Exécution → Modifier le type d'exécution → GPU T4*. Sur un processeur classique, DistilBERT prendrait plusieurs heures.

## 0. Installation, imports et GPU

- `transformers` (Hugging Face) : DistilBERT, son tokenizer et l'outil d'entraînement `Trainer`
- `datasets` : format de données attendu par `Trainer`
- `torch` (PyTorch) : construction et entraînement du LSTM
- `scikit-learn` : découpage train/validation et métriques, **les mêmes que dans le notebook 03**

La cellule suivante vérifie que le GPU est bien activé. La graine aléatoire (`GRAINE = 42`) rend les résultats reproductibles d'une exécution à l'autre.

In [ ]:
%pip install -q transformers accelerate datasets

In [ ]:
import os
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 150)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU :", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU T4")

In [ ]:
# Graine aléatoire fixée pour des résultats reproductibles
GRAINE = 42
random.seed(GRAINE)
np.random.seed(GRAINE)
torch.manual_seed(GRAINE);

## 1. Chargement des données

Mêmes fichiers que le notebook 03, chargés depuis GitHub sur Colab.

La variable `N_MAX` permet de **tester rapidement** tout le notebook sur un petit échantillon (par exemple `N_MAX = 2000`) avant de lancer l'entraînement complet. Pour les résultats finaux, elle doit valoir `None`.

In [ ]:
DOSSIER_LOCAL = Path("../data/processed")
URL_GITHUB = "https://raw.githubusercontent.com/Akerman64/ag-news-classifier/main/data/processed"

source = DOSSIER_LOCAL if DOSSIER_LOCAL.exists() else URL_GITHUB
train_df = pd.read_csv(f"{source}/train_clean.csv.gz")
test_df = pd.read_csv(f"{source}/test_clean.csv.gz")

# Pour un test rapide du notebook, mettre par exemple N_MAX = 2000 (None = tout le dataset)
N_MAX = None
if N_MAX:
    train_df = train_df.sample(N_MAX, random_state=GRAINE).reset_index(drop=True)
    test_df = test_df.sample(N_MAX // 4, random_state=GRAINE).reset_index(drop=True)

print("Train :", train_df.shape, "| Test :", test_df.shape)

In [ ]:
# Correspondance numéro ↔ nom de catégorie (0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech)
id2label = train_df.drop_duplicates("label").set_index("label")["label_text"].sort_index().to_dict()
label2id = {v: k for k, v in id2label.items()}
categories = list(id2label.values())
id2label

In [ ]:
# Jeu de validation (10 % du train) : pour suivre l'apprentissage et choisir la meilleure époque.
# Le jeu de test n'est utilisé qu'une seule fois, à la fin.
train_part, val_part = train_test_split(train_df, test_size=0.1, stratify=train_df["label"], random_state=GRAINE)
train_part, val_part = train_part.reset_index(drop=True), val_part.reset_index(drop=True)
print("Entraînement :", len(train_part), "| Validation :", len(val_part), "| Test :", len(test_df))

- Les catégories sont codées de 0 à 3 (0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech) : les réseaux de neurones travaillent avec des numéros. `id2label` permet de retrouver les noms.
- On met de côté **10 % du train comme jeu de validation** : **106 500 articles d'entraînement**, **11 834 de validation**, et toujours **7 425 de test**.
- La validation sert à **suivre l'apprentissage époque par époque** et à garder la meilleure version du modèle. Le test n'est utilisé qu'une seule fois, à la fin, pour une évaluation honnête.
- `stratify` conserve la même proportion de chaque catégorie dans les deux parties.

## 2. Outils communs d'évaluation

Deux fonctions partagées par les deux modèles :
- `taille_dossier_mo` : taille du modèle sauvegardé sur le disque
- `enregistrer_resultat` : calcule l'accuracy et le F1 macro sur le test, et ajoute une ligne au tableau des résultats

In [ ]:
resultats = []
predictions = {}

def taille_dossier_mo(chemin):
    """Taille totale d'un fichier ou d'un dossier, en Mo."""
    chemin = Path(chemin)
    if chemin.is_file():
        return chemin.stat().st_size / 1e6
    return sum(f.stat().st_size for f in chemin.rglob("*") if f.is_file()) / 1e6

def enregistrer_resultat(nom, y_pred, temps_entrainement, taille_mo):
    predictions[nom] = y_pred
    resultats.append({
        "modele": nom,
        "accuracy": accuracy_score(test_df["label"], y_pred),
        "f1_macro": f1_score(test_df["label"], y_pred, average="macro"),
        "temps_entrainement_s": temps_entrainement,
        "taille_mo": taille_mo,
    })
    r = resultats[-1]
    print(f"{nom} : accuracy = {r['accuracy']:.4f}, F1 macro = {r['f1_macro']:.4f}, "
          f"entraînement = {temps_entrainement:.0f} s, taille = {taille_mo:.1f} Mo")

## 3. LSTM bidirectionnel

### 3.1 Vocabulaire et encodage des textes

Le LSTM ne lit pas une matrice TF-IDF mais une **suite de mots dans l'ordre**. Chaque mot est remplacé par un numéro (son rang dans le vocabulaire) :
- `MAX_MOTS = 30 000` : on garde les 30 000 mots les plus fréquents ; les autres deviennent `<unk>` (inconnu, numéro 1)
- `MAX_LEN = 64` : chaque texte est coupé ou complété par des 0 (`<pad>`) pour faire exactement 64 numéros, car le réseau traite les textes par lots de même taille

Le LSTM utilise la colonne `text_clean`, comme les modèles classiques.

In [ ]:
# Longueur des textes nettoyés (en mots) pour choisir la longueur des séquences
longueurs = train_part["text_clean"].str.split().str.len()
print(longueurs.describe(percentiles=[0.5, 0.95, 0.99]).round(1))

In [ ]:
MAX_MOTS = 30_000   # taille du vocabulaire
MAX_LEN = 64        # longueur maximale d'une séquence (en mots)

# Vocabulaire construit sur le jeu d'entraînement uniquement
compteur = Counter(mot for texte in train_part["text_clean"] for mot in texte.split())
vocab = {"<pad>": 0, "<unk>": 1}
for mot, _ in compteur.most_common(MAX_MOTS - 2):
    vocab[mot] = len(vocab)

print("Mots distincts dans le train :", len(compteur))
print("Taille du vocabulaire retenu :", len(vocab))

In [ ]:
def encoder(textes):
    """Transforme chaque texte en une suite de numéros de mots, complétée par des 0 jusqu'à MAX_LEN."""
    sequences = np.zeros((len(textes), MAX_LEN), dtype=np.int64)
    longueurs = np.zeros(len(textes), dtype=np.int64)
    for i, texte in enumerate(textes):
        ids = [vocab.get(mot, 1) for mot in texte.split()][:MAX_LEN]
        sequences[i, :len(ids)] = ids
        longueurs[i] = max(len(ids), 1)
    return torch.tensor(sequences), torch.tensor(longueurs)

def chargeur(df, melanger):
    sequences, longueurs = encoder(df["text_clean"])
    donnees = TensorDataset(sequences, longueurs, torch.tensor(df["label"].values))
    return DataLoader(donnees, batch_size=256, shuffle=melanger)

train_loader = chargeur(train_part, melanger=True)
val_loader = chargeur(val_part, melanger=False)
test_loader = chargeur(test_df, melanger=False)

print("Exemple :", train_part["text_clean"][0])
print("Encodé  :", encoder(train_part["text_clean"][:1])[0][0].tolist())

Sur le jeu d'entraînement complet :
- Les textes nettoyés font **24 mots en médiane**, et **99 % font moins de 43 mots** (maximum 97). Avec `MAX_LEN = 64`, seuls **0,08 %** des textes sont tronqués.
- Il y a **52 406 mots distincts** ; les 30 000 plus fréquents couvrent **98,9 % des occurrences**. Les mots écartés sont des mots rares (noms propres, fautes) qui apparaissent une ou deux fois.
- Dans l'exemple encodé, les 0 à la fin sont le remplissage (*padding*) jusqu'à 64.

### 3.2 Architecture du réseau

Le réseau a 3 couches :
1. **Embedding** : chaque mot devient un vecteur de 128 nombres, appris pendant l'entraînement. Des mots de sens proche (*match*, *game*) finissent par avoir des vecteurs proches.
2. **LSTM bidirectionnel** : lit la phrase de gauche à droite **et** de droite à gauche, en gardant une « mémoire » de ce qu'il a lu. Contrairement au TF-IDF, il tient compte de **l'ordre des mots**.
3. **Couche de sortie** : transforme le résumé de la phrase (256 nombres = 128 par sens de lecture) en 4 scores, un par catégorie.

Le **dropout** (30 %) désactive aléatoirement des neurones pendant l'entraînement pour limiter le surapprentissage. `pack_padded_sequence` fait ignorer au LSTM les 0 de remplissage.

In [ ]:
class ClassifieurLSTM(nn.Module):
    def __init__(self, taille_vocab, dim_embedding=128, dim_cachee=128, nb_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, dim_embedding, padding_idx=0)
        self.lstm = nn.LSTM(dim_embedding, dim_cachee, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.sortie = nn.Linear(2 * dim_cachee, nb_classes)

    def forward(self, sequences, longueurs):
        x = self.embedding(sequences)
        # pack_padded_sequence : le LSTM ignore les 0 de remplissage
        x = pack_padded_sequence(x, longueurs.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(x)
        h = torch.cat([h[-2], h[-1]], dim=1)   # dernier état caché : sens avant + sens arrière
        return self.sortie(self.dropout(h))

modele_lstm = ClassifieurLSTM(len(vocab)).to(device)
print(modele_lstm)
print("Nombre de paramètres :", sum(p.numel() for p in modele_lstm.parameters()))

Avec le vocabulaire complet (30 000 mots), le réseau compte environ **4,1 millions de paramètres**, dont **3,84 millions dans la couche d'embedding** (30 000 mots × 128). Le LSTM lui-même n'en a qu'environ 264 000.

### 3.3 Entraînement

L'entraînement se fait par **époques** : une époque = un passage complet sur les 106 500 articles, par lots de 256. À chaque lot :
1. le réseau prédit les catégories ;
2. la **perte** (*cross-entropy*) mesure l'écart avec les vraies catégories ;
3. l'optimiseur **Adam** ajuste les poids pour réduire cette perte.

Après chaque époque, on calcule le F1 macro sur la **validation**. On garde les poids de la meilleure époque, et on s'arrête dès que la validation se dégrade (**arrêt anticipé**) : c'est le signe que le réseau commence à apprendre le train par cœur.

In [ ]:
def predire_lstm(modele, loader):
    modele.eval()
    sorties = []
    with torch.no_grad():
        for sequences, longueurs, _ in loader:
            sorties.append(modele(sequences.to(device), longueurs).argmax(dim=1).cpu())
    return torch.cat(sorties).numpy()

def entrainer_une_epoque(modele, loader, optimiseur, critere):
    modele.train()
    perte_totale = 0
    for sequences, longueurs, labels in loader:
        optimiseur.zero_grad()
        perte = critere(modele(sequences.to(device), longueurs), labels.to(device))
        perte.backward()
        optimiseur.step()
        perte_totale += perte.item() * len(labels)
    return perte_totale / len(loader.dataset)

In [ ]:
EPOQUES_LSTM = 5
optimiseur = torch.optim.Adam(modele_lstm.parameters(), lr=1e-3)
critere = nn.CrossEntropyLoss()

historique_lstm = []
meilleur_f1, meilleur_etat = 0, None
debut = time.time()

for epoque in range(1, EPOQUES_LSTM + 1):
    perte = entrainer_une_epoque(modele_lstm, train_loader, optimiseur, critere)
    f1_val = f1_score(val_part["label"], predire_lstm(modele_lstm, val_loader), average="macro")
    historique_lstm.append({"epoque": epoque, "perte_train": perte, "f1_validation": f1_val})
    print(f"Époque {epoque} : perte = {perte:.4f}, F1 validation = {f1_val:.4f}")

    if f1_val > meilleur_f1:   # on garde les poids de la meilleure époque
        meilleur_f1 = f1_val
        meilleur_etat = {k: v.clone() for k, v in modele_lstm.state_dict().items()}
    else:                      # arrêt anticipé dès que la validation se dégrade
        print("→ arrêt anticipé")
        break

temps_lstm = time.time() - debut
modele_lstm.load_state_dict(meilleur_etat)
print(f"Temps d'entraînement : {temps_lstm:.0f} s")

In [ ]:
hist = pd.DataFrame(historique_lstm).set_index("epoque")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hist["perte_train"].plot(ax=axes[0], marker="o", color="indianred", title="Perte sur l'entraînement")
hist["f1_validation"].plot(ax=axes[1], marker="o", color="steelblue", title="F1 macro sur la validation")
for ax in axes:
    ax.set_xlabel("Époque")
    ax.set_xticks(hist.index)
plt.tight_layout()
plt.show()

*(à compléter après l'exécution complète sur Colab)*

*À observer : la perte doit diminuer à chaque époque. Le F1 de validation monte puis plafonne ou redescend, et l'époque où il est maximal est celle qu'on garde.*

### 3.4 Évaluation sur le jeu de test

Le modèle (poids + vocabulaire) est sauvegardé pour mesurer sa taille, puis évalué une seule fois sur le jeu de test.

In [ ]:
DOSSIER_SORTIE = Path("modeles_deep")
DOSSIER_SORTIE.mkdir(exist_ok=True)

chemin_lstm = DOSSIER_SORTIE / "lstm.pt"
torch.save({"etat": modele_lstm.state_dict(), "vocab": vocab, "max_len": MAX_LEN}, chemin_lstm)

enregistrer_resultat("LSTM", predire_lstm(modele_lstm, test_loader), temps_lstm, taille_dossier_mo(chemin_lstm))

*(à compléter après l'exécution complète sur Colab)*

*À comparer avec le notebook 03 : le LSTM fait-il mieux que le SVM linéaire (F1 = 0.911) ? Pour quel temps d'entraînement et quelle taille ?*

## 4. DistilBERT (fine-tuning)

### 4.1 Tokenisation

**DistilBERT** est une version allégée de BERT, un modèle de langage pré-entraîné par Google sur des milliards de mots (Wikipedia et des livres) : il « connaît » déjà l'anglais. On ajoute une couche de classification par-dessus et on ré-entraîne l'ensemble sur AG News (**fine-tuning**). C'est du **transfert d'apprentissage**.

Différences importantes avec les autres modèles :
- DistilBERT lit la colonne **`text`** (débruitée mais **non prétraitée**) : il a appris sur des phrases complètes, avec leurs stopwords et leur ponctuation, qui portent du sens pour lui.
- Il découpe les textes en **sous-mots** (*tokens*) : un mot inconnu est découpé en morceaux connus. Dans l'exemple, *DiMarco* devient `dim`, `##ar`, `##co` : il n'y a jamais de mot « inconnu ».

In [ ]:
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)

NOM_MODELE = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(NOM_MODELE)

# DistilBERT lit la colonne "text" (texte débruité mais NON prétraité) : il a besoin des phrases complètes
exemple = train_part["text"][0]
print(exemple)
print(tokenizer.tokenize(exemple)[:30])

In [ ]:
# Longueur des textes en tokens, pour vérifier que 128 tokens suffisent
nb_tokens = [len(ids) for ids in tokenizer(train_part["text"].sample(min(5000, len(train_part)), random_state=GRAINE).tolist())["input_ids"]]
print(pd.Series(nb_tokens).describe(percentiles=[0.5, 0.95, 0.99]).round(1))
print(f"Textes de plus de 128 tokens : {np.mean(np.array(nb_tokens) > 128) * 100:.2f} %")

In [ ]:
MAX_TOKENS = 128

def vers_dataset(df):
    ds = Dataset.from_pandas(df[["text", "label"]].rename(columns={"label": "labels"}))
    return ds.map(lambda lot: tokenizer(lot["text"], truncation=True, max_length=MAX_TOKENS),
                  batched=True, remove_columns=["text"])

train_ds, val_ds, test_ds = vers_dataset(train_part), vers_dataset(val_part), vers_dataset(test_df)
train_ds

Sur un échantillon de 5 000 articles du train :
- **48 tokens en médiane**, 99 % des textes en dessous de **93 tokens** (maximum 178).
- Seuls **0,12 %** des textes dépassent **128 tokens** et sont tronqués : `MAX_TOKENS = 128` suffit largement. Une limite plus haute ralentirait l'entraînement pour rien.
- Les textes sont tokenisés une fois pour toutes avec `map` ; `labels` est le nom de colonne attendu par `Trainer`.

### 4.2 Entraînement

On charge DistilBERT avec une **tête de classification** à 4 sorties. Le message `MISSING` au chargement est normal : cette tête est nouvelle et ses poids sont initialisés au hasard, c'est justement ce que le fine-tuning va apprendre.

Réglages de l'entraînement (`TrainingArguments`) :
- **2 époques** : un modèle pré-entraîné apprend vite, plus d'époques mènent au surapprentissage
- `learning_rate=2e-5` : un pas d'apprentissage **très petit**, pour ajuster le modèle sans « effacer » ce qu'il sait déjà de l'anglais
- `batch_size=32` : 32 articles par lot
- évaluation à chaque époque sur la validation, et `load_best_model_at_end` pour garder la meilleure époque selon le F1 macro
- `fp16` : calcul en demi-précision sur GPU, environ 2 fois plus rapide
- `DataCollatorWithPadding` complète chaque lot à la longueur de son plus long texte (et non à 128), ce qui accélère encore l'entraînement

In [ ]:
modele_bert = AutoModelForSequenceClassification.from_pretrained(
    NOM_MODELE, num_labels=len(categories), id2label=id2label, label2id=label2id
)

def calculer_metriques(eval_pred):
    logits, labels = eval_pred
    y_pred = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, y_pred), "f1_macro": f1_score(labels, y_pred, average="macro")}

EPOQUES_BERT = 2
arguments = TrainingArguments(
    output_dir="distilbert_checkpoints",
    num_train_epochs=EPOQUES_BERT,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,          # garde la meilleure époque (F1 de validation)
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=torch.cuda.is_available(),       # calcul en demi-précision sur GPU : ~2x plus rapide
    logging_steps=200,
    report_to="none",
    seed=GRAINE,
)

trainer = Trainer(
    model=modele_bert,
    args=arguments,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),   # complète chaque lot à la longueur du plus long texte
    compute_metrics=calculer_metriques,
)

In [ ]:
debut = time.time()
trainer.train()
temps_bert = time.time() - debut
print(f"Temps d'entraînement : {temps_bert:.0f} s")

In [ ]:
journal = pd.DataFrame(trainer.state.log_history)
evals = journal.dropna(subset=["eval_f1_macro"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]
evals

*(à compléter après l'exécution complète sur Colab)*

*À observer : l'évolution de la perte et du F1 de validation entre les 2 époques, et le temps total d'entraînement (à comparer aux quelques secondes des modèles classiques).*

### 4.3 Évaluation sur le jeu de test

Le modèle et son tokenizer sont sauvegardés (la taille mesurée est celle de ce dossier), puis évalués une seule fois sur le jeu de test.

In [ ]:
chemin_bert = DOSSIER_SORTIE / "distilbert"
trainer.save_model(chemin_bert)
tokenizer.save_pretrained(chemin_bert)

sortie_test = trainer.predict(test_ds)
enregistrer_resultat("DistilBERT", sortie_test.predictions.argmax(axis=1), temps_bert, taille_dossier_mo(chemin_bert))

*(à compléter après l'exécution complète sur Colab)*

*À comparer : le gain de F1 par rapport au SVM linéaire (0.911) justifie-t-il un modèle d'environ 270 Mo, au-dessus de la limite de 250 Mo de Vercel ?*

## 5. Comparaison LSTM / DistilBERT

Tableau récapitulatif, matrices de confusion côte à côte et rapport détaillé de DistilBERT.

In [ ]:
comparaison = pd.DataFrame(resultats).set_index("modele")
comparaison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, nom in zip(axes, comparaison.index):
    matrice = confusion_matrix(test_df["label"], predictions[nom], normalize="true") * 100
    sns.heatmap(matrice, annot=True, fmt=".1f", cmap="Blues", cbar=False, ax=ax,
                xticklabels=categories, yticklabels=categories)
    ax.set_title(f"{nom} (% de chaque vraie catégorie)")
    ax.set_xlabel("Catégorie prédite")
    ax.set_ylabel("Vraie catégorie")
plt.tight_layout()
plt.show()

In [ ]:
print(classification_report(test_df["label"], predictions["DistilBERT"], target_names=categories, digits=4))

*(à compléter après l'exécution complète sur Colab)*

*À observer : la confusion Business ↔ Sci/Tech, principale difficulté des modèles classiques, est-elle réduite ?*

## 6. Export des résultats

Les métriques sont exportées dans `metrics_deep.csv`, au format attendu par le notebook 05. Sur Colab, le fichier est téléchargé automatiquement ; il faut ensuite le placer dans le dossier `models/` du dépôt.

Les modèles entraînés sont trop lourds pour GitHub (près de 270 Mo pour DistilBERT). Pour les conserver, passer `SAUVEGARDER_SUR_DRIVE = True` : ils seront copiés sur Google Drive.

In [ ]:
comparaison.round(4).to_csv("metrics_deep.csv")
print(open("metrics_deep.csv").read())

In [ ]:
# Sur Colab : télécharge metrics_deep.csv, à placer ensuite dans le dossier models/ du dépôt
try:
    from google.colab import files
    files.download("metrics_deep.csv")
except ImportError:
    print("Hors Colab : copier metrics_deep.csv dans models/")

In [ ]:
# Optionnel : sauvegarder les modèles entraînés sur Google Drive (trop lourds pour GitHub)
SAUVEGARDER_SUR_DRIVE = False

if SAUVEGARDER_SUR_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    shutil.copytree(DOSSIER_SORTIE, "/content/drive/MyDrive/ag-news-classifier/modeles_deep", dirs_exist_ok=True)
    print("Modèles copiés sur Google Drive")

## 7. Bilan

*(à compléter après l'exécution complète sur Colab)*